# PINN 3 — 2D Poisson equation

$$\Delta u = \frac{\partial^2u}{\partial x^2}+\frac{\partial^2u}{\partial y^2}
= -2\pi^2\sin(\pi x)\sin(\pi y)\quad\text{on }[0,1]^2,\qquad u=0 \text{ on } \partial\Omega,$$

exact solution $u=\sin(\pi x)\sin(\pi y)$.

New: a 2D spatial domain, four boundary edges, the full Laplacian, and a **residual (skip-
connection) MLP**.

| | |
|---|---|
| Problem class | `neural_pde_solvers.pinn.Poisson2D` |
| Config | `configs/pinn/poisson2d.yaml` |
| CLI | `pinn-run configs/pinn/poisson2d.yaml` |

In [ ]:
# --- Setup -------------------------------------------------------------------
# Locally: `pip install -e ".[pinn]"` from the repository root.
# On Google Colab the package is installed from GitHub automatically.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("neural_pde_solvers") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "neural-pde-solvers[pinn] @ git+https://github.com/tvh0ng/neural-pde-solvers.git",
        ]
    )

import matplotlib.pyplot as plt
import numpy as np
import torch

from neural_pde_solvers import set_seed, setup_logging
from neural_pde_solvers.pinn import (
    MLP,
    PINNTrainer,
    TrainerConfig,
    count_parameters,
    evaluate,
    grid,
    predict,
    resolve_device,
)
from neural_pde_solvers.pinn.plotting import plot_loss_history

setup_logging()
set_seed(42)
device = resolve_device("auto")
QUICK = os.environ.get("PINN_QUICK") == "1"  # shortened run used by CI
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. Problem and training points
100 boundary points per edge; interior collocation points are re-drawn every epoch.

In [ ]:
from neural_pde_solvers.pinn import Poisson2D

problem = Poisson2D(n_bc=400).to(device)
X_col = problem.sample_collocation(5000).detach().cpu()
X_bc = problem.X_bc.cpu()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_col[:, 0], X_col[:, 1], s=1, alpha=0.3, label="collocation")
ax.scatter(X_bc[:, 0], X_bc[:, 1], s=8, c="tab:red", label="boundary")
ax.set(aspect="equal", xlabel="x", ylabel="y", title="Training points")
ax.legend(loc="upper right")
plt.show()

## 2. Residual MLP

Hidden layers compute $h \leftarrow \tanh(Wh+b) + h$; skip connections ease gradient flow
in deeper networks.

In [ ]:
model = MLP(in_dim=2, width=64, depth=6, activation="tanh", residual=True).to(device)
print(f"trainable parameters: {count_parameters(model):,}")

## 3. Training

In [ ]:
config = TrainerConfig(
    epochs=500 if QUICK else 8000,
    lr=1e-3,
    n_collocation=5000,
    scheduler="plateau",
    scheduler_kwargs=dict(factor=0.5, patience=500, min_lr=1e-6),
    log_every=1000,
)
history = PINNTrainer(model, problem, config).fit()

## 4. Results

In [ ]:
from neural_pde_solvers.pinn.plotting import plot_prediction_vs_exact_2d

mesh, X = grid(problem, 101)
u_pred, u_exact = predict(model, X), problem.exact(X)
print(evaluate(model, problem, 101))
fig, axes = plot_prediction_vs_exact_2d(mesh, u_pred, u_exact)
for ax in axes:
    ax.set_aspect("equal")
plt.show()

In [ ]:
Xm, Ym = mesh
fig = plt.figure(figsize=(15, 5))
ax1 = fig.add_subplot(1, 3, 1, projection="3d")
ax1.plot_surface(Xm, Ym, u_pred.reshape(Xm.shape), cmap="viridis")
ax1.set_title("PINN prediction")
ax2 = fig.add_subplot(1, 3, 2, projection="3d")
ax2.plot_surface(Xm, Ym, u_exact.reshape(Xm.shape), cmap="viridis")
ax2.set_title("Exact solution")
plot_loss_history(history, ax=fig.add_subplot(1, 3, 3))
fig.tight_layout()
plt.show()

## Key takeaways

1. 2D problems need more collocation points (5 000 here vs. 100 in 1D).
2. Each boundary edge must be sampled; corners are where errors usually peak.
3. `laplacian(u, X)` sums second derivatives over all input columns.

**Try:** a non-zero boundary condition, another source term, or an L-shaped domain
(override `sample_collocation` and the boundary points).
Next: **PINN 4 — inverse problem**.